# Page Object Model: Escenarios de Usuario

## Introducción

El Page Object Model (POM) es un patrón de diseño que ayuda a organizar las pruebas E2E separando la lógica de la página de la lógica de las pruebas.

---

## 1. ¿Qué es el Page Object Model?

El POM crea una clase por cada página (o componente) de la aplicación que:
- Encapsula los locators de los elementos
- Proporciona métodos para interactuar con la página
- Abstrae la implementación de la UI

### Ventajas del POM
- **Reutilización**: Los métodos pueden usarse en múltiples pruebas
- **Mantenibilidad**: Si cambia la UI, solo se actualiza el Page Object
- **Legibilidad**: Las pruebas leen como escenarios de usuario
- **Separación de concerns**: Lógica de UI separada de lógica de prueba

## 2. Implementación de TaskPage

A continuación implementamos un Page Object para la página de tareas.

In [ ]:
from playwright.sync_api import Page, expect


class TaskPage:
    """Page Object para la página de gestión de tareas."""
    
    def __init__(self, page: Page):
        self.page = page
        
        # Locators
        self.title_input = page.locator('[data-testid="task-title-input"]')
        self.description_input = page.locator('[data-testid="task-description-input"]')
        self.create_button = page.locator('[data-testid="create-task-button"]')
        self.task_list = page.locator('[data-testid="task-list"]')
        self.task_items = page.locator('[data-testid="task-item"]')
        self.task_title = page.locator('[data-testid="task-title"]')
        self.task_description = page.locator('[data-testid="task-description"]')
        self.completed_badge = page.locator('[data-testid="completed-badge"]')
        self.complete_button = page.locator('[data-testid="complete-button"]')
        self.delete_button = page.locator('[data-testid="delete-button"]')
        self.empty_state = page.locator('[data-testid="empty-state"]')
        self.empty_message = page.locator('[data-testid="empty-message"]')
    
    def goto(self):
        """Navegar a la página de tareas."""
        self.page.goto('http://localhost:5000')
        return self
    
    def create_task(self, title: str, description: str = ""):
        """Crear una nueva tarea."""
        self.title_input.fill(title)
        if description:
            self.description_input.fill(description)
        self.create_button.click()
        return self
    
    def complete_task(self, task_index: int = 0):
        """Marcar una tarea como completada."""
        self.complete_button.nth(task_index).click()
        return self
    
    def delete_task(self, task_index: int = 0):
        """Eliminar una tarea."""
        self.delete_button.nth(task_index).click()
        return self
    
    def get_task_count(self):
        """Obtener el número de tareas en la lista."""
        return self.task_items.count()
    
    def get_task_title(self, index: int = 0):
        """Obtener el título de una tarea específica."""
        return self.task_title.nth(index).text_content()
    
    def is_task_completed(self, index: int = 0):
        """Verificar si una tarea está completada."""
        return self.completed_badge.nth(index).is_visible()
    
    def is_empty(self):
        """Verificar si la lista de tareas está vacía."""
        return self.empty_state.is_visible()

print("TaskPage implementada exitosamente")

## 3. Pruebas usando TaskPage

Ahora las pruebas son más legibles y mantenibles.

In [ ]:
from playwright.sync_api import sync_playwright, expect


def test_create_task_with_pom():
    """Prueba de creación de tarea usando Page Object."""
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        
        # Usar el Page Object
        task_page = TaskPage(page)
        task_page.goto()
        
        # Crear tarea
        task_page.create_task('Tarea con POM', 'Descripción de prueba')
        
        # Verificar
        expect(task_page.task_items).to_have_count(1)
        expect(task_page.task_title).to_contain_text('Tarea con POM')
        
        browser.close()

# test_create_task_with_pom()
print("Prueba con POM definida")

## 4. Flujo completo de usuario

El POM permite escribir flujos complejos de manera natural.

In [ ]:
def test_complete_user_flow():
    """Prueba el flujo completo: crear → completar → eliminar."""
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        
        task_page = TaskPage(page)
        task_page.goto()
        
        # 1. Crear tarea
        task_page.create_task('Tarea de flujo completo')
        expect(task_page.task_items).to_have_count(1)
        
        # 2. Verificar que no está completada
        assert not task_page.is_task_completed(0)
        
        # 3. Completar tarea
        task_page.complete_task(0)
        
        # 4. Verificar que está completada
        assert task_page.is_task_completed(0)
        expect(task_page.completed_badge).to_contain_text('✓ Completada')
        
        # 5. Eliminar tarea
        task_page.delete_task(0)
        
        # 6. Verificar que ya no existe
        expect(task_page.task_items).to_have_count(0)
        assert task_page.is_empty()
        
        browser.close()

# test_complete_user_flow()
print("Flujo completo definido")

## 5. Comparación: Sin POM vs Con POM

### Sin Page Object
```python
def test_without_pom(page):
    page.goto('http://localhost:5000')
    page.fill('[data-testid="task-title-input"]', 'Test')
    page.click('[data-testid="create-task-button"]')
    expect(page.locator('[data-testid="task-item"]')).to_have_count(1)
    page.click('[data-testid="complete-button"]')
    expect(page.locator('[data-testid="completed-badge"]')).to_be_visible()
    page.click('[data-testid="delete-button"]')
    expect(page.locator('[data-testid="task-item"]')).to_have_count(0)
```

### Con Page Object
```python
def test_with_pom(page):
    task_page = TaskPage(page)
    task_page.goto()
    task_page.create_task('Test')
    expect(task_page.task_items).to_have_count(1)
    task_page.complete_task(0)
    assert task_page.is_task_completed(0)
    task_page.delete_task(0)
    expect(task_page.task_items).to_have_count(0)
```

El código con POM es más legible, reutilizable y mantenible.

---

## Ejercicio

1. Implementa la clase `TaskPage` en un archivo separado (ej. `tests/page_objects.py`)
2. Reescribe las pruebas de la Parte 4 usando `TaskPage`
3. Agrega un flujo completo: crear → completar → verificar → eliminar → verificar ausencia